# Prerequistes

- In order to download ALL card scans we'll setup mtg-bulk-database with postgres and then update ./.env with credentials:

https://github.com/JakeTurner616/mtg-bulk-database

In [ ]:
# Cell 1: Install required libraries (if not already installed)
%pip install numpy opencv-python h5py faiss-cpu matplotlib
%pip install psycopg2-binary python-dotenv aiohttp tqdm nest_asyncio

## Descriptor Extraction Pipeline

This workflow extracts SIFT descriptors from MTG card images and stores them very efficiently:

- Uses only `.h5`, `.index`, and `id_map.json` — no SQLite or extra mappings.
- Appends to `candidate_features.h5` without overwriting existing data.
- Automatically batches descriptors to temp `.npy` files for low memory usage.
- Builds a FAISS index only when new data is added or index is missing.
- Fully resumable and idempotent — reruns won't duplicate or erase data.

# Feature extraction
- Preprocess  with CLAHE adjustments to the luminance channel, and convert back to the RGB color space
- Extract features using SIFT, apply RootSIFT normalization, and Product Quantization into an Inverted File Index
- ~~Generate a index_to_card.txt for robust searching of keypoints with RANSAC~~
- Add to existing model files instead of creating new ones each time

In [ ]:
import os
import json
import gc
import numpy as np
import psycopg2
import psycopg2.extras
import h5py
import faiss
import psutil
import tempfile
from dotenv import load_dotenv
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from feature_worker import process_record

# ---------------------------
# CONFIG
# ---------------------------
load_dotenv('.env')
DB_USER = os.getenv("POSTGRES_USER", "mtguser")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "mtgpass")
DB_NAME = os.getenv("POSTGRES_DB", "mtgdb")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")

H5_FEATURES_FILE = 'resources/run/candidate_features.h5'
FAISS_INDEX_FILE = 'resources/run/faiss_ivf.index'
ID_MAP_FILE = 'resources/run/id_map.json'
EVAL_SAMPLE_COUNT = 10
MAX_WORKERS = 4
MEMORY_UPDATE_EVERY = 10
FAISS_BATCH_SIZE = 5000

raw_limit = os.getenv("DATASET_LIMIT")
dataset_limit = int(raw_limit) if raw_limit and raw_limit.isdigit() else None

def get_memory_usage():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

# ---------------------------
# LOAD RECORDS
# ---------------------------
def load_card_records():
    limit_clause = f"LIMIT {dataset_limit}" if dataset_limit else ""
    conn = psycopg2.connect(
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
        host=DB_HOST, port=DB_PORT
    )
    query = f"""
        SELECT id AS scryfall_id,
               COALESCE(image_uris->>'png', image_uris->>'large') AS image_url,
               0 AS face_index
        FROM cards
        WHERE layout::text NOT IN ('art_series', 'scheme', 'plane', 'phenomenon')
        AND games @> '["paper"]'
        AND lang = 'en'
        AND digital = false
        AND (promo IS NULL OR promo = false)
        AND image_uris IS NOT NULL
        {limit_clause}
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(query)
        return cur.fetchall()

card_records = load_card_records()

# ---------------------------
# HDF5 SETUP
# ---------------------------
if os.path.exists(H5_FEATURES_FILE):
    with h5py.File(H5_FEATURES_FILE, 'a') as hf:
        processed_ids = set(hf.keys())
else:
    with h5py.File(H5_FEATURES_FILE, 'w'):
        pass
    processed_ids = set()

new_records = [r for r in card_records if r['scryfall_id'] not in processed_ids]
tqdm.write(f"Total records: {len(card_records)} | New to process: {len(new_records)}")

# ---------------------------
# EXTRACTION + INDEXING
# ---------------------------
eval_samples = []
descriptor_files = []
batch_descriptors = []
id_map = []
descriptor_count = 0

with tempfile.TemporaryDirectory() as temp_dir:
    with h5py.File(H5_FEATURES_FILE, 'a') as hf, ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        progress = tqdm(total=len(new_records), desc="Extracting features", unit="img")
        for i, result in enumerate(executor.map(process_record, new_records)):
            if not result:
                progress.update(1)
                continue

            scryfall_id = result["scryfall_id"]
            image_url = result["image_url"]
            keypoints = result["keypoints"]
            descriptors = np.array(result["descriptors"], dtype=np.float16)

            card_grp = hf.create_group(scryfall_id) if scryfall_id not in hf else hf[scryfall_id]
            feat_grp = card_grp.create_group(f"feature_{len(card_grp)}")
            feat_grp.create_dataset("descriptors", data=descriptors, compression="gzip", compression_opts=3, chunks=True)
            feat_grp.create_dataset("keypoints", data=[json.dumps(keypoints)],
                                    dtype=h5py.string_dtype(encoding="utf-8"),
                                    compression="gzip", compression_opts=3, chunks=True)
            feat_grp.attrs["image_url"] = image_url  # ✅ Store image_url in HDF5

            batch_descriptors.extend([d.astype(np.float32) for d in descriptors])
            id_map.extend([scryfall_id] * len(descriptors))
            descriptor_count += len(descriptors)

            if len(eval_samples) < EVAL_SAMPLE_COUNT:
                eval_samples.append({
                    "scryfall_id": scryfall_id,
                    "image_url": image_url,
                    "face_index": result["face_index"]
                })

            if descriptor_count >= FAISS_BATCH_SIZE:
                path = os.path.join(temp_dir, f"descriptors_{len(descriptor_files)}.npy")
                np.save(path, np.vstack(batch_descriptors))
                descriptor_files.append(path)
                batch_descriptors.clear()
                descriptor_count = 0
                gc.collect()

            if i % MEMORY_UPDATE_EVERY == 0:
                progress.set_postfix(mem=f"{get_memory_usage():.1f} MB")
            progress.update(1)
        progress.close()

    # Final flush
    if batch_descriptors:
        path = os.path.join(temp_dir, f"descriptors_{len(descriptor_files)}.npy")
        np.save(path, np.vstack(batch_descriptors))
        descriptor_files.append(path)
        batch_descriptors.clear()
        gc.collect()

    # Save ID map
    if id_map:
        with open(ID_MAP_FILE, "w") as f:
            json.dump(id_map, f)
        tqdm.write(f"✅ Saved ID map with {len(id_map)} entries to {ID_MAP_FILE}")

    # Build or skip FAISS index
    if descriptor_files:
        tqdm.write(f"[INFO] Building FAISS index from {len(descriptor_files)} descriptor files.")
        mmap_refs = [np.load(path, mmap_mode='r') for path in descriptor_files]
        train_data = mmap_refs[0]
        dim = train_data.shape[1]

        if len(id_map) < 10000:
            index = faiss.IndexFlatL2(dim)
            tqdm.write(f"[INFO] Using IndexFlatL2 for {len(id_map)} descriptors.")
        else:
            quantizer = faiss.IndexFlatL2(dim)
            index = faiss.IndexIVFPQ(quantizer, dim, 100, 8, 8)
            index.nprobe = 10
            index.train(train_data[:10000])
            tqdm.write("[INFO] Trained IVF-PQ index.")

        for mmap in tqdm(mmap_refs, desc="Adding descriptors to FAISS"):
            index.add(np.array(mmap))

        faiss.write_index(index, FAISS_INDEX_FILE)
        tqdm.write(f"✅ FAISS index written to disk with {index.ntotal} descriptors.")

# Inference and Test accuracy

In [ ]:
import os
import cv2
import json
import faiss
import h5py
import random
import numpy as np
import requests
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG + LOAD INDEX & ID MAP
# ---------------------------
H5_FEATURES_FILE = "resources/run/candidate_features.h5"
FAISS_INDEX_FILE = "resources/run/faiss_ivf.index"
ID_MAP_FILE = "resources/run/id_map.json"

index = faiss.read_index(FAISS_INDEX_FILE)

with open(ID_MAP_FILE, "r") as f:
    id_map = json.load(f)

# ---------------------------
# DEFINE MULTIPLE VALID IDS
# ---------------------------
sample = {
    "scryfall_ids": [
        "3394cefd-a3c6-4917-8f46-234e441ecfb6",
        "710160a6-43b4-4ba7-9dcd-93e01befc66f"
    ],
    "image_url": "https://cards.scryfall.io/large/front/3/3/3394cefd-a3c6-4917-8f46-234e441ecfb6.jpg?1592487887",
    "face_index": 0
}
image_url = sample["image_url"]
ground_truth_ids = set(sample["scryfall_ids"])  # Convert to set for fast lookup

# ---------------------------
# DESCRIPTOR EXTRACTION + VISUALIZATION
# ---------------------------
def extract_query_descriptors_from_url(image_url, max_features=100, visualize=False):
    response = requests.get(image_url, timeout=10)
    image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Failed to decode image from URL: {image_url}")

    image = cv2.resize(image, (256, 256))
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    L_clahe = clahe.apply(L)
    lab_clahe = cv2.merge((L_clahe, A, B))
    gray = cv2.cvtColor(cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2GRAY)

    sift = cv2.SIFT_create(nfeatures=max_features)
    keypoints, descriptors = sift.detectAndCompute(gray, None)

    if descriptors is not None and len(keypoints) > max_features:
        sorted_kp_des = sorted(zip(keypoints, descriptors), key=lambda x: -x[0].response)
        keypoints, descriptors = zip(*sorted_kp_des[:max_features])
        keypoints, descriptors = list(keypoints), np.array(descriptors)

    if descriptors is not None:
        eps = 1e-7
        descriptors = descriptors / (descriptors.sum(axis=1, keepdims=True) + eps)
        descriptors = np.sqrt(descriptors).astype(np.float32)

    if visualize and keypoints is not None:
        image_with_kp = cv2.drawKeypoints(image, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(image_with_kp, cv2.COLOR_BGR2RGB))
        plt.title("Query Image with SIFT Keypoints")
        plt.axis("off")
        plt.show()

    return descriptors

# ---------------------------
# FAISS MATCHING
# ---------------------------
def predict_card_from_url(image_url, index, id_map, top_k=5, visualize=False):
    descriptors = extract_query_descriptors_from_url(image_url, visualize=visualize)
    if descriptors is None:
        return None, []

    D, I = index.search(descriptors, top_k)

    prediction_counts = {}
    for indices in I:
        for i in indices:
            if i < len(id_map):
                card_id = id_map[i]
                prediction_counts[card_id] = prediction_counts.get(card_id, 0) + 1

    sorted_preds = sorted(prediction_counts.items(), key=lambda x: -x[1])
    return sorted_preds[0][0] if sorted_preds else None, sorted_preds[:top_k]

# ---------------------------
# RUN INFERENCE
# ---------------------------
top_pred, top_matches = predict_card_from_url(image_url, index, id_map, top_k=10, visualize=True)

print(f"Query image URL: {image_url}")
print(f"Valid ground truth IDs: {list(ground_truth_ids)}")
print(f"Top prediction:          {top_pred}")
print("Top matches:")
for match_id, score in top_matches:
    print(f"  {match_id}: {score}")

if top_pred in ground_truth_ids:
    print("✅ Prediction is CORRECT (matched one of the valid IDs).")
else:
    print("❌ Prediction is INCORRECT (did not match any valid IDs).")


## Package the resources.zip:

In [ ]:
import zipfile
import os

# ---------------------------
# CONFIG
# ---------------------------
OUTPUT_ZIP = 'resourcesV4.zip'
FILES_TO_INCLUDE = [
    'resources/run/candidate_features.h5',
    'resources/run/faiss_ivf.index',
    'resources/run/id_map.json',
]

# ---------------------------
# ZIP PACKAGING
# ---------------------------
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in FILES_TO_INCLUDE:
        if os.path.exists(file_path):
            arcname = os.path.relpath(file_path, start='resources')
            zipf.write(file_path, arcname=os.path.join('resources', arcname))
            print(f"✔️ Added: {file_path}")
        else:
            print(f"⚠️ Skipped (not found): {file_path}")

print(f"\n📦 Packaged inference archive saved to: {OUTPUT_ZIP}")
